# Actividad 4: Scraping, de la página al CSV

Este cuaderno baja tres series a `raw/`, de la más fácil a la más trabajosa:

| | fuente | método | serie |
|---|---|---|---|
| 1 | **FRED** | API: se pide la serie por su código y llega en JSON | precio del zinc |
| 2 | **INE** | scraping: leer la página, encontrar el enlace, ordenar el Excel | PIB trimestral (el ejemplo de la presentación) |
| 3 | **BCB** | la misma receta, con un archivo que cambia de nombre cada semana | reservas internacionales |

**Modo sin conexión.** Si un sitio no responde, cada descarga usa automáticamente una copia
guardada en `data/sin_conexion/`. Para trabajar todo el cuaderno sin internet, ponga
`SIN_CONEXION = True` en la celda de parámetros.

Las celdas con `<SU CÓDIGO AQUÍ: ...>` son para completar. La solución está en `sols/act4_scraping.ipynb`.

In [ ]:
# ===========================================================================
# CARGA DE LA CONFIGURACIÓN  --  editar config.py, NO esta celda.
# Deja `cfg` (los parámetros del país) y `P` (las rutas estándar).
# ===========================================================================
import os, sys, importlib.util

CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

PATH_RAW  = P["raw"]      # aquí se escriben los CSV descargados
PATH_DATA = P["data"]     # aquí están las copias para el modo sin conexión

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})")
print(f"         raw  : {PATH_RAW}")
print(f"         data : {PATH_DATA}")

In [ ]:
# ===========================================================================
# Librerías, parámetros y dos funciones de apoyo.
# ===========================================================================
import re, json, unicodedata, warnings
from datetime import date
from io import BytesIO
from urllib.parse import urljoin, unquote

import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

# ---- parámetros ------------------------------------------------------------
SIN_CONEXION = False      # True = no se consulta internet; se usan las copias locales
PATH_COPIAS  = os.path.join(PATH_DATA, "sin_conexion")
FUENTES_CSV  = os.path.join(PATH_RAW, "_fuentes.csv")

# Varias oficinas de estadística responden 403 al "user-agent" de Python.
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

os.makedirs(PATH_COPIAS, exist_ok=True)
MODO = {}                 # copia -> ("en línea" | "copia local", fecha), para el registro de fuentes


def traer(url, copia, timeout=60, **kw):
    # Pide `url` y devuelve el contenido (bytes).
    # Si responde, guarda una copia en data/sin_conexion/<copia>. Si no responde,
    # o si SIN_CONEXION = True, devuelve esa copia.
    ruta = os.path.join(PATH_COPIAS, copia)
    if not SIN_CONEXION:
        try:
            try:
                r = requests.get(url, headers=UA, timeout=timeout, **kw)
            except requests.exceptions.SSLError:
                # el certificado del INE ha estado incompleto otras veces
                r = requests.get(url, headers=UA, timeout=timeout, verify=False, **kw)
            print(f"  {r.status_code} | {r.headers.get('content-type', '')[:45]} | "
                  f"{len(r.content):,} bytes")
            if r.ok:
                with open(ruta, "wb") as f:
                    f.write(r.content)
                MODO[copia] = ("en línea", date.today().isoformat())
                return r.content
        except requests.exceptions.RequestException as e:
            print(f"  sin respuesta: {type(e).__name__}")
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No hay copia local de {copia} en {PATH_COPIAS}")
    print(f"  -> se usa la copia local {copia}")
    MODO[copia] = ("copia local", date.fromtimestamp(os.path.getmtime(ruta)).isoformat())
    with open(ruta, "rb") as f:
        return f.read()


def registrar(archivo, fuente, url, serie, copia):
    # Anota en raw/_fuentes.csv de dónde salió un archivo y qué contiene.
    # Este registro alimenta los metadatos de la base (actividad 5).
    fila = pd.DataFrame([{
        "archivo": archivo, "fuente": fuente, "url": url,
        "descargado": MODO[copia][1], "modo": MODO[copia][0],
        "obs": int(serie.notna().sum()),
        "desde": serie.dropna().index.min().date(), "hasta": serie.dropna().index.max().date()}])
    if os.path.exists(FUENTES_CSV):
        viejo = pd.read_csv(FUENTES_CSV)
        fila = pd.concat([viejo[viejo["archivo"] != archivo], fila], ignore_index=True)
    fila.to_csv(FUENTES_CSV, index=False)
    print(f"  registrado en _fuentes.csv: {archivo} | {fila.iloc[-1]['desde']} a {fila.iloc[-1]['hasta']}")


print("SIN_CONEXION =", SIN_CONEXION)

---
# 1. Una API: FRED

El método más fácil. Una API responde directamente con datos estructurados (JSON): no hay HTML que
recorrer ni Excel que ordenar. Se le pide **una serie** con **un código** y devuelve fechas y valores.
FRED pide una **clave gratuita y personal** (https://fredaccount.stlouisfed.org/apikeys) que va en `config.py`,
en `FRED_API_KEY`. **No se comparte ni se sube a GitHub.** Sin clave, la celda usa la copia local.

Serie: `PZINCUSDM`, precio mundial del zinc (USD por tonelada), el principal mineral que exporta Bolivia.

In [ ]:
SERIE = "PZINCUSDM"
CLAVE = cfg.FRED_API_KEY or os.environ.get("FRED_API_KEY", "")

URL_FRED = "https://api.stlouisfed.org/fred/series/observations"
parametros = {"series_id": SERIE, "api_key": CLAVE, "file_type": "json"}

if CLAVE:
    datos = traer(URL_FRED, f"fred_{SERIE}.json", params=parametros)
else:
    print("sin FRED_API_KEY en config.py")
    _antes, SIN_CONEXION = SIN_CONEXION, True
    datos = traer(URL_FRED, f"fred_{SERIE}.json")
    SIN_CONEXION = _antes

obs = json.loads(datos)["observations"]
zinc = pd.DataFrame(obs)[["date", "value"]]
zinc["date"] = pd.to_datetime(zinc["date"])
zinc = pd.to_numeric(zinc.set_index("date")["value"], errors="coerce").rename(f"fred_{SERIE}").dropna()

zinc.to_csv(os.path.join(PATH_RAW, "fred_zinc.csv"))
registrar("fred_zinc.csv", "FRED", f"{URL_FRED}?series_id={SERIE}", zinc, f"fred_{SERIE}.json")

zinc.loc["2010":].plot(figsize=(10, 3.5), title="Precio del zinc, USD por tonelada (FRED)")
plt.show()

---
# 2. Scraping: el PIB trimestral del INE

Sin API, los datos hay que sacarlos de la página web. Es el ejemplo de la presentación, en seis pasos:

| paso | qué se hace |
|---|---|
| 1 · Mirar | abrir la página en el navegador e inspeccionar el enlace |
| 2 · Pedir | traer el HTML con `requests` |
| 3 · Encontrar | listar los enlaces y elegir **por su título completo** |
| 4 · Descargar | bajar el Excel y abrirlo sin interpretar (`header=None`) |
| 5 · Ordenar | convertir la hoja en una serie con fechas |
| 6 · Guardar y comprobar | escribir el CSV crudo, anotar la fuente y validar con algo conocido |

## Paso 1 · Mirar la página

Abra en el navegador:

**https://www.ine.gob.bo/referencia2017/pib_trimestral.html**

1. Ubique el cuadro *«BOLIVIA: MEDIDAS DE VOLUMEN ENCADENADAS DEL PRODUCTO INTERNO BRUTO POR GRUPOS DE ACTIVIDAD ECONÓMICA…»*.
2. Clic derecho sobre el enlace → **Inspeccionar**.
3. Observe la etiqueta `<a>`: el atributo `href` termina en `/download`, **no** en `.xlsx`. El único
   dato que identifica el cuadro es su **texto**.

Las preguntas para guiar la inspección: ¿la información está en una tabla HTML (`<table>`) o en archivos?
¿Qué tienen en común todos los enlaces de descarga?

## Paso 2 · Pedir la página

`traer()` hace la consulta e imprime el **código de estado** (200 = bien), el tipo de contenido y el tamaño.

In [ ]:
URL_INE = "https://www.ine.gob.bo/referencia2017/pib_trimestral.html"

html = traer(URL_INE, "ine_pib_trimestral.html")

# Lo que llega es texto: el mismo HTML que se ve con "Inspeccionar"
print(html[:300].decode("utf-8", errors="replace"))

## Paso 3 · Encontrar el enlace

`BeautifulSoup` convierte el HTML en un árbol que se puede recorrer. `find_all("a", href=True)` devuelve
todos los enlaces; nos quedamos con los de descarga.

In [ ]:
sopa = BeautifulSoup(html, "lxml")

enlaces = []
for a in sopa.find_all("a", href=True):
    if "/download" in a["href"]:
        texto = a.get_text(" ", strip=True)
        enlaces.append((texto, urljoin(URL_INE, a["href"])))

print(f"{len(enlaces)} cuadros descargables\n")
for i, (texto, _) in enumerate(enlaces):
    print(f"[{i:2d}] {texto[:105]}")

Buscar por **palabras clave** no alcanza: muchos títulos dicen «volumen encadenado» y «producto».
Y comparar el texto **literal** también falla: el título del PIB nominal tiene dos espacios seguidos
(`BRUTO  POR`). La función `clave()` quita tildes, pasa a minúsculas y deja un solo espacio entre palabras.

In [ ]:
def clave(s):
    # Sin tildes, en minúsculas y con un solo espacio entre palabras.
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return " ".join(s.lower().split())


# Tres maneras de buscar el PIB nominal
nominal = "BOLIVIA: PRODUCTO INTERNO BRUTO POR GRUPOS DE ACTIVIDAD ECONÓMICA"

literal  = [t for t, h in enlaces if t.startswith(nominal)]
palabras = [t for t, h in enlaces if "VOLUMEN ENCADENAD" in t and "PRODUCTO" in t]
con_clave = [t for t, h in enlaces if clave(t).startswith(clave(nominal))]

print(f"literal: {len(literal)}   palabras clave (volumen encadenado): {len(palabras)}   "
      f"con clave(): {len(con_clave)}")

In [ ]:
# El PIB REAL: el cuadro de volumen encadenado (referencia 2017)
TITULO_REAL = "BOLIVIA: MEDIDAS DE VOLUMEN ENCADENADAS DEL PRODUCTO INTERNO BRUTO POR GRUPOS"

hits = [href for texto, href in enlaces if clave(texto).startswith(clave(TITULO_REAL))]
assert len(hits) == 1, f"{len(hits)} coincidencias: revise el título contra la lista de arriba"
URL_REAL = hits[0]
print(URL_REAL)

## Paso 4 · Descargar y abrir sin interpretar

`header=None` le dice a pandas que **no adivine** encabezados: la hoja se ve tal como es, con sus filas
de título, celdas vacías y notas al pie.

In [ ]:
contenido = traer(URL_REAL, "ine_T_02.01.xlsx", timeout=120)

raw = pd.read_excel(BytesIO(contenido), header=None)
print("forma de la hoja:", raw.shape)
raw.iloc[6:20, :7]

## Paso 5 · Ordenar: de la hoja a una serie

Lo que hay que resolver, visto arriba:

* el año está sólo en el **primero** de sus cuatro trimestres (celda combinada en Excel) y a veces dice `2025 (p)`;
* los trimestres están en números romanos y con un espacio al final (`'I '`);
* las filas se buscan **por su etiqueta**, nunca por su número: si el INE agrega una fila, el código sigue funcionando.

Convención del taller: cada trimestre se fecha el **primer día de su último mes** (2017T1 = 2017-03-01).

In [ ]:
ROMANO = {"I": 3, "II": 6, "III": 9, "IV": 12}


def hoja_a_serie(raw, etiqueta, nombre):
    # Hoja trimestral del INE -> serie con fechas.
    etiq = raw[1].astype(str).str.strip()                         # columna B: las etiquetas
    fila = etiq[etiq.str.startswith("ACTIVIDAD")].index[0]        # fila de los años
    anio = raw.iloc[fila, 2:].ffill().astype(str).str[:4]         # '2025 (p)' -> '2025'
    mes  = raw.iloc[fila + 1, 2:].astype(str).str.strip().map(ROMANO)
    ok   = mes.notna()                                            # columnas con trimestre
    fechas = pd.to_datetime(anio[ok] + "-" + mes[ok].astype(int).astype(str) + "-01")

    valores = raw.loc[etiq.str.startswith(etiqueta), 2:].iloc[0][ok]
    return pd.Series(valores.astype(float).values, name=nombre,
                     index=pd.DatetimeIndex(fechas, name="date"))


pib = hoja_a_serie(raw, "PRODUCTO INTERNO BRUTO", "gdp")
print(f"{len(pib)} trimestres, {pib.index.min().date()} a {pib.index.max().date()}")
pib.tail()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
pib.plot(ax=ax[0], marker="o", ms=3)
ax[0].set_title("Bolivia: PIB real trimestral (millones de Bs encadenados, ref. 2017)")
pib.pct_change(4).mul(100).plot(ax=ax[1], color="firebrick", marker="o", ms=3)
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_title("variación interanual, %")
plt.tight_layout(); plt.show()

## Paso 6 · Guardar y comprobar

El archivo crudo se guarda **sin editar**: ni ajustes, ni tasas, ni recortes. Todo eso viene después,
en la actividad 5. Y se anota de dónde salió en `raw/_fuentes.csv`.

In [ ]:
pib.to_csv(os.path.join(PATH_RAW, "ine_gdp.csv"))
registrar("ine_gdp.csv", "INE", URL_REAL, pib, "ine_T_02.01.xlsx")

### Ejercicio: el PIB nominal y el deflactor

Repita la receta con el cuadro **nominal** (*«BOLIVIA: PRODUCTO INTERNO BRUTO POR GRUPOS DE ACTIVIDAD
ECONÓMICA…»*, en millones de bolivianos corrientes). La hoja tiene el mismo diseño, así que `hoja_a_serie()`
sirve tal cual: sólo hay que dar el título del cuadro y la fórmula del deflactor.

**Comprobación.** El deflactor implícito (nominal ÷ real × 100) debe estar cerca de 100 en 2017 y **subir**
hasta ~145 a fines de 2025. Si baja, los dos cuadros están al revés.

In [ ]:
TITULO_NOMINAL = <SUBSTITUIR> #### COMPLETAR

hits = [href for texto, href in enlaces if clave(texto).startswith(clave(TITULO_NOMINAL))]
assert len(hits) == 1, f"{len(hits)} coincidencias"
URL_NOMINAL = hits[0]

raw_nom = pd.read_excel(BytesIO(traer(URL_NOMINAL, "ine_T_01.01.xlsx", timeout=120)), header=None)
pib_nom = hoja_a_serie(raw_nom, <COMPLETAR> , "gdp") #### COMPLETAR

pib_nom.to_csv(os.path.join(PATH_RAW, "ine_gdp_nominal.csv"))
registrar("ine_gdp_nominal.csv", "INE", URL_NOMINAL, pib_nom, "ine_T_01.01.xlsx")
pib_nom.tail(5)

In [ ]:
deflactor = (pib_nom / pib * 100).dropna()
print(f"deflactor: {deflactor.iloc[0]:.1f} en {deflactor.index[0].date()} -> "
      f"{deflactor.iloc[-1]:.1f} en {deflactor.index[-1].date()}")
assert deflactor.iloc[-1] > deflactor.iloc[0], "el deflactor baja: ¿cuadros al revés?"

deflactor.plot(figsize=(10, 3.5), marker="o", ms=3, title="Deflactor implícito del PIB (nominal / real x 100)")
plt.show()

---
# 3. Boletín semanal del BCB

Dos diferencias con el INE:

* **El nombre del archivo cambia cada semana** (`Semanal 38_2026.xlsx`, `Semanal 39_2026.xlsx`…), con espacios
  irregulares y caracteres codificados en el enlace (`Semanal%2038_2026.xlsx`). Se decodifica con `unquote`
  y se toma la **semana más alta**.
* **Las series se buscan por nombre**, en la columna D (índice 3). Las fechas ocupan dos filas: fin de mes en la
  fila 3 para la historia y fechas diarias en la fila 4 para las últimas semanas.

Página: **https://www.bcb.gob.bo/?q=estad-sticas-semanales**

In [ ]:
URL_BCB = <COMPLETAR> ################## COMPLETAR
sopa_bcb = BeautifulSoup(traer( <COMPLETAR> , "bcb_estadisticas_semanales.html"), "lxml") ############# COMPLETAR

boletines = []
for a in sopa_bcb.find_all("a", href=True):
    href = unquote( <COMPLETAR> )                          ########################### COMPLETAR

    # Regular Expression:
    m = re.search(r"Semanal\s*(\d{1,2})\s*_\s*(\d{4})", href, re.I)
    if m and href.lower().endswith(".xlsx"):
        boletines.append((int(m.group(2)), int(m.group(1)), urljoin(URL_BCB, a["href"])))

print(f"{len(boletines)} boletines en la página")
anio_b, semana, URL_BOLETIN = max(boletines)
print(f"el más reciente: semana {semana} de {anio_b}\n{URL_BOLETIN}")

Veamos la lista ```boletines```:

Y ahora que tenemos la URL mas reciente, limpiamos:

In [ ]:
bcb = pd.read_excel(BytesIO(traer(URL_BOLETIN, "bcb_semanal.xlsx", timeout=120)), header=None)
bcb

In [ ]:
# Las fechas: diarias en la fila 4 (índice 3) y, donde falten, fin de mes en la fila 3 (índice 2)
fechas = (pd.to_datetime(bcb.iloc[3], errors="coerce")
            .combine_first(pd.to_datetime(bcb.iloc[2], errors="coerce")))
con_fecha = fechas.notna()
print(f"{int(con_fecha.sum())} columnas con fecha, {fechas[con_fecha].min().date()} a {fechas[con_fecha].max().date()}")

# Las etiquetas están en la columna D (índice 3). Buscar las que interesan:
etiquetas = bcb[3].astype(str).str.strip()
etiquetas[etiquetas.str.contains("reservas", case=False)]

In [ ]:
BUSCAR = <COMPLETAR> ###### Cual es la variable de interes?

fila = etiquetas[etiquetas.str.startswith(BUSCAR)].index[0]
rin = pd.Series(pd.to_numeric(bcb.loc[fila, con_fecha], errors="coerce").values,
                index=pd.DatetimeIndex(fechas[con_fecha].values, name="date"),
                name="bcb_rin").dropna().sort_index()
rin = rin[~rin.index.duplicated(keep="last")]

rin.to_csv(os.path.join(PATH_RAW, "bcb_rin.csv"))
registrar("bcb_rin.csv", "BCB", URL_BOLETIN, rin, "bcb_semanal.xlsx")

rin.loc["2014":].plot(figsize=(10, 3.5), title=f"BCB: {etiquetas[fila]}")
plt.show()

---
## Lo que quedó en `raw/`

Cuatro archivos crudos (zinc, PIB real, PIB nominal, reservas) y el registro `_fuentes.csv`: de dónde salió cada uno, cuándo se bajó, si vino de internet
o de la copia local, y qué periodo cubre. Esa información es la que después alimenta los **metadatos** de la base.

In [ ]:
pd.read_csv(FUENTES_CSV)

---
## Extra: cuando los datos están en la página

Si la página muestra una tabla HTML (`<table>`), `pd.read_html` la convierte en DataFrame en una línea. La página
del PIB no tiene ninguna `<table>` (los datos están en los Excel), así que el ejemplo usa un HTML escrito a mano.
En páginas en español, `decimal=","` y `thousands="."` evitan leer mal «1.234,5».

In [ ]:
from io import StringIO

print("tablas <table> en la página del PIB:", len(sopa.find_all("table")))

ejemplo = '''
<table>
  <tr><th>mes</th><th>valor</th></tr>
  <tr><td>2026-07</td><td>1.234,5</td></tr>
  <tr><td>2026-08</td><td>1.301,2</td></tr>
</table>'''

tablas = pd.read_html(StringIO(ejemplo), decimal=",", thousands=".")   # una lista de DataFrames
tablas[0]